## 1. Dados & Preparação

Nesta seção, apresento de forma clara e reprodutível como a base de clientes é carregada e preparada para análise. O objetivo é garantir **qualidade dos dados**, **padronização** e **transparência** antes de partir para a exploração e as visualizações.

**O que será feito:**
- **Origem dos dados:** anexo um *dataset* privado no Kaggle e leio o arquivo `.csv` diretamente do ambiente do notebook.
- **Leitura robusta:** detecto automaticamente o separador mais adequado (`,`, `;`, `\t`, `|`) para evitar erros comuns de importação.
- **Padronização de nomes:** converto os nomes das colunas para *snake_case* (minúsculas e `_`), facilitando leitura e escrita de código.
- **Conversão monetária (pt-BR → numérico):** campos como `limite_credito` e `valor_transacoes_12m` são transformados do formato “1.234,56” para `1234.56`, permitindo cálculos confiáveis.
- **Categorização ordenada:** quando presente, `salario_anual` é tratada como categoria ordenada (faixas salariais em ordem lógica).
- **Validação inicial:** apresento dimensões do *dataset*, uma amostra de linhas e os tipos de dados de cada coluna.

**Saídas esperadas desta etapa:**
- Confirmação do *shape* (linhas × colunas);
- *Preview* das primeiras linhas para checagem visual;
- Tipos de dados coerentes com o conteúdo (numéricos, categóricos, etc.).

> Observação: as transformações são realizadas em memória, preservando o arquivo original e assegurando que o processo possa ser reproduzido em qualquer execução do notebook.


In [ ]:
import pandas as pd, numpy as np, glob, re

DATA_PATH = "<cole_aqui_o_caminho_do_csv>"
if DATA_PATH == "<cole_aqui_o_caminho_do_csv>":
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not candidates:
        raise SystemExit("Nenhum CSV encontrado. Use Add data → anexe seu dataset privado.")
    DATA_PATH = candidates[0]

df = None
for sep in [",", ";", "\t", "|"]:
    try:
        t = pd.read_csv(DATA_PATH, sep=sep, engine="python")
        if t.shape[1] == 1 and t.shape[0] > 1:
            continue  # ignora leituras que viram 1 coluna só
        df = t
        break
    except Exception:
        pass
if df is None:
    raise RuntimeError("Falha ao ler o CSV com separadores padrão.")

def to_snake(name: str) -> str:
    name = re.sub(r"[^\w]+", "_", str(name).strip().lower())
    name = re.sub(r"_+", "_", name).strip("_")
    return name
df.rename(columns={c: to_snake(c) for c in df.columns}, inplace=True)

def to_float_ptbr(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

for col in ["limite_credito", "valor_transacoes_12m"]:
    if col in df.columns and df[col].dtype == "object":
        df[col] = to_float_ptbr(df[col])

if "salario_anual" in df.columns:
    ordem = ["menos que $40K", "$40K - $60K", "$60K - $80K", "$80K - $120K", "$120K - $150K", "mais que $150K"]
    df["salario_anual"] = pd.Categorical(df["salario_anual"], categories=ordem, ordered=True)

if "default" in df.columns:
    df["default"] = pd.to_numeric(df["default"], errors="coerce").fillna(0).astype(int)

print("df pronto:", df.shape)


## 2. Exploração & Qualidade

Nesta etapa, avalio a estrutura do conjunto de dados para identificar **tipos de variáveis**, **valores ausentes**, **possíveis duplicidades** e **pontos atípicos (outliers)**. O objetivo é assegurar a **confiabilidade** das análises e orientar decisões de tratamento quando necessário.

**Nesta seção verifico:**
- Perfil das colunas (tipo, cardinalidade, % de nulos);
- Duplicatas (linha inteira) e colunas com alta **unicidade** (candidatas a chave);
- Resumo descritivo das variáveis numéricas;
- Sinalização de outliers pelo critério **IQR**.


In [ ]:
# 2. Exploração & Qualidade — perfil, nulos, duplicatas, outliers
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype, is_categorical_dtype, is_datetime64_any_dtype

# Garantir que 'default' esteja numérica binária, se existir
if "default" in df.columns:
    df["default"] = pd.to_numeric(df["default"], errors="coerce").fillna(0).astype(int)

def classif(s: pd.Series) -> str:
    if is_numeric_dtype(s): 
        return "numérica"
    if is_datetime64_any_dtype(s): 
        return "data"
    if is_categorical_dtype(s) or s.dtype == "object": 
        return "categórica"
    return "categórica"

# Perfil das colunas
linhas = len(df)
profile_rows = []
for c in df.columns:
    s = df[c]
    profile_rows.append({
        "coluna": c,
        "tipo": classif(s),
        "n_unique": s.nunique(dropna=True),
        "%_nulos": round(s.isna().mean()*100, 2),
    })
profile_df = pd.DataFrame(profile_rows).sort_values(["tipo", "coluna"]).reset_index(drop=True)
display(profile_df)

# Nulos (top 20)
nul = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(nul.to_frame("%_nulos").head(20))

# Duplicatas e unicidade de colunas
dup_total = df.duplicated().sum()
print(f"Duplicatas (linha inteira): {dup_total}")

unicidade = (df.nunique(dropna=False) / linhas * 100).round(2).sort_values(ascending=False)
display(unicidade.to_frame("%_unicidade").head(10))  # colunas >~99% são candidatas a chave

# Resumo das numéricas
num_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
if num_cols:
    display(df[num_cols].describe(percentiles=[.25, .5, .75]).T.round(3))

# Sinalização de outliers por IQR
out_rows = []
for c in num_cols:
    s = df[c].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low_iqr, high_iqr = q1 - 1.5*iqr, q3 + 1.5*iqr
    pct_iqr = ((s < low_iqr) | (s > high_iqr)).mean()*100
    out_rows.append({
        "coluna": c,
        "lim_iqr": (round(low_iqr, 3), round(high_iqr, 3)),
        "%_outliers_iqr": round(pct_iqr, 2),
    })
out_df = pd.DataFrame(out_rows).sort_values("%_outliers_iqr", ascending=False)
display(out_df)


## 3. Visualizações-chave

Nesta seção, apresento gráficos concisos para entender a distribuição de valores, a composição por categorias e relações entre variáveis. As figuras geradas também são salvas na pasta `plots/` (Output) para eventual uso em relatórios.


In [ ]:
# 3. Visualizações-chave — geração de gráficos
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pasta de saída
OUTDIR = "plots"
os.makedirs(OUTDIR, exist_ok=True)

# Métrica principal (fixa) e colunas auxiliares
PRIMARY_METRIC = "valor_transacoes_12m" if "valor_transacoes_12m" in df.columns else (
                 "limite_credito" if "limite_credito" in df.columns else None)
CAT_A = "sexo"             # contagem por categoria
CAT_B = "escolaridade"     # contagem por categoria
CAT_C = "tipo_cartao"      # agregações por categoria
TARGET = "default"         # taxa média por categoria

def save_show(fig, fname):
    path = os.path.join(OUTDIR, fname)
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.show()
    print(f"✔ salvo em: {path}")

# 1) Distribuição da métrica principal
if PRIMARY_METRIC:
    s = pd.to_numeric(df[PRIMARY_METRIC], errors="coerce").dropna()

    # Histograma
    fig = plt.figure()
    plt.hist(s, bins=30)
    plt.title(f"Distribuição — {PRIMARY_METRIC}")
    plt.xlabel(PRIMARY_METRIC)
    plt.ylabel("Frequência")
    save_show(fig, f"hist_{PRIMARY_METRIC}.png")

    # Boxplot
    fig = plt.figure()
    plt.boxplot(s, vert=True, showfliers=True)
    plt.title(f"Boxplot — {PRIMARY_METRIC}")
    plt.ylabel(PRIMARY_METRIC)
    save_show(fig, f"box_{PRIMARY_METRIC}.png")

# 2) Barras: contagem por categorias principais
for CAT in [c for c in [CAT_A, CAT_B] if c in df.columns]:
    vc = df[CAT].value_counts(dropna=False).head(10)
    fig = plt.figure()
    plt.bar(vc.index.astype(str), vc.values)
    plt.title(f"Top-10 {CAT} — contagem de linhas")
    plt.xlabel(CAT)
    plt.ylabel("Contagem")
    plt.xticks(rotation=45, ha="right")
    save_show(fig, f"bar_top10_count_{CAT}.png")

# 3) Barras: soma da métrica por tipo de cartão
if PRIMARY_METRIC and (CAT_C in df.columns):
    tmp = (df[[CAT_C, PRIMARY_METRIC]]
           .assign(**{PRIMARY_METRIC: pd.to_numeric(df[PRIMARY_METRIC], errors="coerce")})
           .dropna(subset=[PRIMARY_METRIC])
           .groupby(CAT_C, dropna=False)[PRIMARY_METRIC]
           .sum()
           .sort_values(ascending=False))
    fig = plt.figure()
    plt.bar(tmp.index.astype(str), tmp.values)
    plt.title(f"Soma de {PRIMARY_METRIC} por {CAT_C}")
    plt.xlabel(CAT_C)
    plt.ylabel(f"Soma de {PRIMARY_METRIC}")
    plt.xticks(rotation=45, ha="right")
    save_show(fig, f"bar_sum_{PRIMARY_METRIC}_by_{CAT_C}.png")

# 4) Taxa média de default por tipo de cartão
if (TARGET in df.columns) and (CAT_C in df.columns):
    y = pd.to_numeric(df[TARGET], errors="coerce")
    tmp = (df.assign(**{TARGET: y})
             .dropna(subset=[TARGET])
             .groupby(CAT_C, dropna=False)[TARGET]
             .mean()
             .sort_values(ascending=False))
    fig = plt.figure()
    plt.bar(tmp.index.astype(str), tmp.values)
    plt.title(f"Taxa de {TARGET} por {CAT_C}")
    plt.xlabel(CAT_C)
    plt.ylabel(f"Taxa de {TARGET}")
    plt.xticks(rotation=45, ha="right")
    save_show(fig, f"bar_rate_{TARGET}_by_{CAT_C}.png")

# 5) Heatmap de correlação (exclui 'id' para não distorcer)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != "id"]
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    fig = plt.figure(figsize=(6,5))
    plt.imshow(corr, interpolation="nearest")
    plt.title("Matriz de Correlação (numéricas)")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)
    plt.colorbar()
    plt.tight_layout()
    save_show(fig, "heatmap_correlacao.png")

# 6) Dispersão: limite_credito × valor_transacoes_12m
if {"limite_credito","valor_transacoes_12m"}.issubset(df.columns):
    x = pd.to_numeric(df["limite_credito"], errors="coerce")
    y = pd.to_numeric(df["valor_transacoes_12m"], errors="coerce")
    m = (~x.isna()) & (~y.isna())
    fig = plt.figure()
    plt.scatter(x[m], y[m], s=8, alpha=0.5)
    plt.title("Dispersão — limite_credito × valor_transacoes_12m")
    plt.xlabel("limite_credito")
    plt.ylabel("valor_transacoes_12m")
    save_show(fig, "scatter_limite_vs_valor.png")

print("\nPronto!")


## 4. Insights (Storytelling)

### 4.1 Panorama geral
- Base analisada com **10.127 observações** e **16 variáveis**.
- **Inadimplência (default)** média de **16,1%**, patamar que exige atenção e monitoramento.
- **Qualidade dos dados**: não há duplicidades; a variável **salario_anual** possui cerca de **18,16%** de valores ausentes.

### 4.2 Perfil do cliente
- **Sexo**: distribuição equilibrada, com leve predominância do público **Feminino** (5.358) em relação ao **Masculino** (4.769).
- **Escolaridade** (amostra das mais frequentes): **mestrado** (3.128), **ensino médio** (2.013), “**na**”/desconhecido (1.519), **sem educação formal** (1.487), **graduação** (1.013).
- **Estado civil** (amostra): **casado** (4.687), **solteiro** (3.943), **divorciado** (748) e “**na**”/desconhecido (749).
- **Idade**: média em torno de **46,3 anos** (distribuição relativamente concentrada no intervalo ~26–73).

### 4.3 Relacionamento e ciclo de vida
- **Tempo de relacionamento**: mediana em **36 meses** (1º–3º quartil: ~31–40 meses), indicando base com **clientes de médio prazo**.
- **Associação forte entre idade e meses de relacionamento** (|r| ≈ **0,79**): clientes mais antigos tendem a ser também os de maior tempo de conta — efeito coerente com maturidade/antiguidade.

### 4.4 Métricas financeiras e uso
- **Limite de crédito**: média **R$ 8.632,44**; mediana **R$ 4.549,42**. Há **cauda longa** e ~**9,7%** de outliers (IQR), o que é esperado em distribuições financeiras.
- **Valor transacionado em 12 meses**: média **R$ 4.404,58**; mediana **R$ 3.899,59**; também com **cauda longa** e ~**8,9%** de outliers (IQR).
- O **histograma** e o **boxplot** sugerem concentração de clientes com **baixo a médio volume** e um grupo menor que concentra **volumes elevados** (efeito de concentração).

### 4.5 Composição e desempenho por categorias
- **Tipo de cartão**: os gráficos de **soma de transações por tipo** e **taxa de default por tipo** evidenciam **diferenças relevantes** entre categorias, úteis para priorização comercial e gestão de risco.
- **Sexo** e **Escolaridade**: a análise de barras revela a distribuição do portfólio por perfis. Essas variáveis são candidatas a **segmentação de campanhas** e **comunicação dirigida**.

### 4.6 Relações entre variáveis
- A **matriz de correlação** (excluindo `id`) indica relações moderadas/pontuais entre variáveis numéricas. 
- No **gráfico de dispersão** *limite × valor transacionado* observa-se **tendência positiva**: **limites maiores** tendem a **acompanhar volumes maiores** (efeito esperado em portfólios de cartão).

> **Síntese:** a carteira apresenta perfil adulto, relacionamento mediano, distribuição financeira com cauda longa e inadimplência na casa de 16%. Há diferenças por tipo de cartão e perfis socioeconômicos que justificam estratégias de segmentação e oferta, combinadas com controles de risco e estímulos de ativação.


## 5. Recomendações

**5.1 Estratégia comercial e de ativação**
- Priorizar ofertas nos **tipos de cartão** que concentram maior volume transacionado, com pacotes de benefícios (ex.: cashback, pontos bônus, parcelamento diferenciado) e campanhas sazonais.
- Construir jornadas de **ativação para baixa movimentação** (isenção de anuidade inicial, bônus na 1ª compra, campanhas “volte a usar”), acompanhando impacto em frequência e ticket.
- Avaliar **ajustes graduais de limite** para perfis com bom histórico, condicionados a comportamento e risco, com “*guardrails*” para evitar sobrealavancagem.

**5.2 Gestão de risco e inadimplência**
- Implementar **painel de alerta precoce** (queda brusca de transações, aumento de meses_inativo_12m, redução de interações) para ações preventivas de retenção/renegociação.
- Desenvolver (ou calibrar) um **score de propensão ao default** usando variáveis comportamentais (ex.: idade, meses_de_relacionamento, iteracoes_12m, qtd_transacoes_12m), avaliando pontos de corte e *trade-offs*.
- Para segmentos com risco acima da média, reforçar **comunicação proativa**, ofertas de **parcelamento** e **renegociação assistida**.

**5.3 Segmentação e personalização**
- Usar **tipo_cartao**, **escolaridade** e **faixas salariais** (quando informadas) como eixos de **comunicação dirigida** e oferta de produtos/serviços com maior aderência.
- Testar **programas educacionais** (uso consciente do crédito, planejamento financeiro) para grupos de maior risco percebido.

**5.4 Governança e qualidade de dados**
- Tratar a lacuna de **salario_anual (~18%)**: padronizar preenchimento, criar rótulo explícito para “não informado” e revisar coleta em canais.
- Manter **dicionário de dados** e **pipeline** de padronização (separador, *encoding*, formatação monetária), garantindo reprodutibilidade e consistência.
- Fixar **identificador único** (`id`) como chave canônica em todos os fluxos.

**5.5 Mensuração e experimentação**
- Monitorar mensalmente **taxa de default**, **volume transacionado** (total e por tipo de cartão), **limite médio** e **concentração** (ex.: participação do decil superior).
- Conduzir **testes A/B** (ex.: incentivos de ativação, limites dinâmicos), com definição prévia de métricas de sucesso e horizonte de observação.


## 6. Conclusão

A análise revela uma carteira com **perfil adulto** (média ≈46 anos), **tempo de relacionamento mediano** (mediana ≈36 meses) e **distribuições financeiras com cauda longa**: poucos clientes concentram volumes elevados de transações e limites mais altos. A **inadimplência média (≈16%)** requer atenção contínua, mas há espaço relevante para **crescimento com rentabilidade** ao focar segmentos e ofertas mais aderentes.

Os gráficos indicam:
- Diferenças claras por **tipo de cartão**, úteis para priorização comercial e calibragem de risco.
- **Associação forte** entre idade e meses de relacionamento (efeito de maturidade).
- **Relação positiva** entre limite e volume transacionado, coerente com a dinâmica de crédito.
- Heterogeneidade por **sexo** e **escolaridade**, que favorece **segmentação e personalização**.

Em síntese, recomenda-se combinar **estratégias de ativação** (para ampliar uso com controle de risco), **segmentação orientada por dados** e **governança de informações** (especialmente sobre salário), sustentadas por **painéis e experimentos** para tomada de decisão baseada em evidências.


## 7. Reprodutibilidade

- **Ambiente:** Notebook Kaggle (CPU), Python 3.x.
- **Origem dos dados:** dataset privado anexado em `/kaggle/input/...`.
- **Processo:** leitura robusta do `.csv`, padronização de nomes (*snake_case*) e conversão monetária pt-BR → numérico; nenhuma alteração é feita no arquivo original.
- **Identificador:** `id` é usado como chave única.
- **Saídas:** as figuras são salvas em `plots/` (aba *Output*).
- **Versões:** cada entrega é registrada via **Save Version** no Kaggle, com descrição do que mudou.
- **Boas práticas:** células organizadas por seções, títulos descritivos, comentários objetivos e métricas expostas antes das conclusões.
